# Bab 7 — Outlier Detection and Handling

Notebook pendamping Bab 7 Level 2.

Notebook ini mengikuti halaman bab di situs. Jalankan sel dari atas ke bawah.

In [ ]:
import os
from pathlib import Path

# Pindah ke folder utama proyek agar "data/mall_customers.csv" ketemu.
if not Path("data").exists():
    os.chdir("../..")

## Langkah 1: Siapkan alat dan muat data

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

DATA_PATH = "data/mall_customers.csv"
df = pd.read_csv(DATA_PATH)

fitur = ["Age", "Annual Income (k$)", "Spending Score (1-100)"]

## Langkah 2: Mendeteksi outlier dengan aturan IQR

In [ ]:
q1 = df[fitur].quantile(0.25)
q3 = df[fitur].quantile(0.75)
iqr = q3 - q1

batas_bawah = q1 - 1.5 * iqr
batas_atas = q3 + 1.5 * iqr

print("Batas bawah:")
print(batas_bawah)
print("\nBatas atas:")
print(batas_atas)

di_luar = (df[fitur] < batas_bawah) | (df[fitur] > batas_atas)
print("\nJumlah outlier per kolom (aturan IQR):")
print(di_luar.sum())

print("\nBaris yang punya outlier:")
print(df[di_luar.any(axis=1)])

## Langkah 3: Mendeteksi outlier dengan z-score

In [ ]:
z = (df[fitur] - df[fitur].mean()) / df[fitur].std()

print("Nilai |z| terbesar per kolom:")
print(z.abs().max().round(2))

print("\nJumlah |z| > 3 per kolom:")
print((z.abs() > 3).sum())

## Langkah 4: Melihat outlier pada grafik

In [ ]:
kolom = "Annual Income (k$)"
is_outlier = di_luar[kolom]

fig, ax = plt.subplots(figsize=(6, 4.5))
ax.scatter(
    df.loc[~is_outlier, kolom],
    df.loc[~is_outlier, "Spending Score (1-100)"],
    s=16,
    label="Normal",
)
ax.scatter(
    df.loc[is_outlier, kolom],
    df.loc[is_outlier, "Spending Score (1-100)"],
    s=60,
    color="red",
    label="Outlier (IQR)",
)
ax.set_xlabel(kolom)
ax.set_ylabel("Spending Score (1-100)")
ax.legend()
plt.show()

## Langkah 5: Pilihan 1: buang baris outlier

In [ ]:
df_buang = df[~di_luar.any(axis=1)]

print("Sebelum:", df.shape)
print("Sesudah:", df_buang.shape)

## Langkah 6: Pilihan 2: batasi nilai (capping)

In [ ]:
df_batas = df.copy()
for k in fitur:
    df_batas[k] = df_batas[k].clip(lower=batas_bawah[k], upper=batas_atas[k])

print("Ukuran:", df_batas.shape)
print("Nilai maksimum Annual Income sebelum:", df[kolom].max())
print("Nilai maksimum Annual Income sesudah :", df_batas[kolom].max())

## Langkah 7: Bandingkan ketiga pilihan

In [ ]:
ringkas = pd.DataFrame(
    {
        "asli": df[kolom].agg(["count", "mean", "std", "max"]),
        "dibuang": df_buang[kolom].agg(["count", "mean", "std", "max"]),
        "dibatasi": df_batas[kolom].agg(["count", "mean", "std", "max"]),
    }
).round(2)

print(ringkas)